In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="reach-vb/jenny_tts_dataset", 
                  repo_type="dataset", local_dir="./jenny_tts_dataset")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 12 files: 100%|██████████| 12/12 [00:04<00:00,  2.57it/s]


'/home/ubuntu/jenny_tts_dataset'

In [3]:
files = glob('jenny_tts_dataset/*/*.parquet')
len(files)

10

In [11]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['transcription'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_jenny"
            })
        
    return data

In [10]:
# data = loop((files[:1], 0))
# data

In [12]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 2098/2098 [03:50<00:00,  9.11it/s]


In [14]:
data[0]

{'audio_filename': 'jenny_tts_dataset_audio/jenny_tts_dataset-data-train-00002-of-00010_0.mp3',
 'text': "So for days on end we followed the tiger's footprints.",
 'speaker': 'jenny_tts_dataset_audio_jenny'}

In [15]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'jenny_tts_dataset_audio/jenny_tts_dataset-data-train-00002-of-00010_0.mp3',
 'text': "So for days on end we followed the tiger's footprints.",
 'speaker': 'jenny_tts_dataset_audio_jenny'}

In [16]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'jenny_tts_dataset')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 107.07ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 1.27MB / 1.27MB,  127kB/s  
Processing Files (1 / 1): 100%|██████████| 1.27MB / 1.27MB,  127kB/s  
New Data Upload: 100%|██████████| 1.27MB / 1.27MB,  127kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.52s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/a5d044a8a584f61e5cd4cb2046819ee5136ea798', commit_message='Upload dataset', commit_description='', oid='a5d044a8a584f61e5cd4cb2046819ee5136ea798', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [17]:
audio_files = [d['audio_filename'] for d in data]

with open('jenny_tts_dataset-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [18]:
!zip -rq jenny_tts_dataset_audio.zip jenny_tts_dataset_audio

In [20]:
# !hf upload malaysia-ai/Multilingual-TTS jenny_tts_dataset_audio.zip --repo-type=dataset

In [24]:
# !zip -rq jenny_tts_dataset_audio_neucodec.zip jenny_tts_dataset_audio_neucodec

In [25]:
# !hf upload malaysia-ai/Multilingual-TTS jenny_tts_dataset_audio_neucodec.zip --repo-type=dataset